# Prompt Evaluation — Part 4: Grounding the Judge with Per-Task Criteria

The remaining weakness of the model judge is that it grades against its own vague, shifting notion of "good." This final notebook fixes that by telling the judge **exactly what to look for on each task**.

**What's new:**

- **Each test case now carries a `solution_criteria` field**, generated alongside the task. It spells out what a correct answer must contain for *that specific* task.
- **`grade_by_model` injects those criteria into the judge prompt** (the `<criteria>` block). The judge is no longer improvising a standard — it's checking the output against a fixed rubric. This makes scores more consistent, more reproducible, and less gameable by verbosity.
- Because the rubric is written *when the task is created* (not at grading time), the standard is fixed in advance — the eval measures the prompt, not the judge's mood.

**The complete recipe (all four notebooks together):**

1. Generate a **dataset** of representative tasks — each with a format and its own success **criteria**.
2. **Run** the prompt under test on every case, constraining output so it's gradable.
3. **Grade** with a hybrid: deterministic code checks for objective correctness + a criteria-grounded model judge for quality.
4. **Average** to a single number you can watch move as you iterate.

Now the score is a real signal: change the prompt, re-run, and see whether it went up or down.

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

**Setup — unchanged.** Same client and model as the rest of the series.

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

**Helpers — unchanged.**

In [3]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

**Changed: each case now carries `solution_criteria`.** The generator writes a per-task rubric *at creation time*, so the standard for "correct" is pinned down before any grading happens — not improvised by the judge mid-run.

In [4]:
# dataset.json is reused across the whole series, BUT this notebook's grader needs the extra
# `solution_criteria` field. So we reuse the existing file only if it has that field — otherwise
# (missing file, or one generated by notebook 002/003) we regenerate it here.
import os


def dataset_has_criteria():
    if not os.path.exists("dataset.json"):
        return False
    with open("dataset.json") as f:
        data = json.load(f)
    return all("solution_criteria" in case for case in data)


if dataset_has_criteria():
    print("dataset.json already exists with solution_criteria — reusing it. Delete the file to regenerate.")
else:
    dataset = generate_dataset()
    with open("dataset.json", "w") as f:
        json.dump(dataset, f, indent=2)
    print("Generated dataset.json (this notebook needs the solution_criteria field).")

dataset.json already exists with solution_criteria — reusing it. Delete the file to regenerate.


**Schema-aware reuse.** This notebook's grader needs `solution_criteria`, so the check reuses the existing `dataset.json` only if every case has that field — otherwise (a file left by notebook 002/003, or none at all) it regenerates. That's why running this notebook standalone is always safe.

In [5]:
# Function to grade a test case + output using a model
import re


def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])

    # LLM-authored JSON is fragile: when the judge echoes the solution's regex or Python,
    # its raw backslashes (\d, \., \w ...) are illegal JSON escapes and json.loads raises
    # "Invalid \escape". If the first parse fails, escape any backslash that isn't a valid
    # JSON escape and try once more.
    try:
        return json.loads(eval_text)
    except json.JSONDecodeError:
        repaired = re.sub(r'\\(?!["\\/bfnrtu])', r"\\\\", eval_text)
        return json.loads(repaired)

**Changed: the judge is now grounded.** The `<criteria>` block feeds each task's rubric into the judge prompt, so it grades against a fixed, task-specific standard instead of its own shifting sense of "good." The payoff: scores are more consistent, more reproducible, and much harder to game with verbosity.

In [6]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

**Prompt under test — unchanged from notebook 003** (constrained to bare code, no commentary).

In [7]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


**Validators — unchanged** from notebook 003. The objective half of the hybrid score.

In [8]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

**Hybrid score — unchanged mechanics**, but both halves are now stronger: a deterministic syntax check *and* a criteria-grounded judge, averaged.

In [9]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

**Scoreboard — unchanged.**

In [10]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.5


**Run it — the complete eval.** Rubric-bearing dataset → constrained run → code check + grounded judge → one comparable number. This is the payoff of the whole series: change the prompt under test, re-run, and trust the score to tell you whether it got better.

In [11]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\nimport json\n\nlog_pattern = r'^(?P<timestamp>\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}\\.\\d{3}Z)\\s+(?P<level>[A-Z]+)\\s+(?P<message>.*)$'\n\ndef parse_cloudwatch_log(log_entry):\n    match = re.match(log_pattern, log_entry)\n    if match:\n        return match.groupdict()\n    return None\n\nlog_entry = \"2024-01-15T10:30:45.123Z ERROR Failed to process request\"\nresult = parse_cloudwatch_log(log_entry)\nprint(json.dumps(result, indent=2))\n",
    "test_case": {
      "task": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message using a regular expression",
      "format": "regex",
      "solution_criteria": "The regex should correctly capture timestamp in ISO 8601 format, log level (INFO, ERROR, WARN, DEBUG), and the remaining message text. Should handle variations in spacing and log formats."
    },
    "score": 8.0,
    "reasoning": "The solution successfully parses the provided example and handles the core require